# Reconstruction audit of the current motion codecs

Evaluate **ground-truth motion → frozen RVQ tokens → reconstructed motion**, using the codec checkpoints referenced by your current gesture-model config. The gesture transformer and speech models are not loaded.

The default comparison is **Scott train versus validation**. The notebook reports rotation, expression and translation errors, optional SMPL-X geometry errors, and code usage at every RVQ stage. It saves the sample IDs and checkpoint hashes beside the results.

Run this notebook in the repository's training Python environment, on the machine containing the BEATX HDF5 cache. Start with `MAX_SAMPLES_PER_SPLIT = 8` to check setup, then use `None` for the complete comparison. No training or W&B run is started.


In [ ]:
# Edit this cell before running.
from pathlib import Path

# None auto-detects the repository from the notebook/kernel working directory.
# Otherwise use an absolute path, e.g. "/home/wenjye/miburi".
REPO_ROOT = None
LM_CONFIG = "configs/gtdm3_teacher_c_face1_cos800_rvq_beatx_scott.yaml"

CUDA_VISIBLE_DEVICES = "2"       # Set to None to preserve the kernel's GPU visibility.
DEVICE = "cuda:0"                # Logical device after GPU selection; "cpu" also works.
PARTS = ("upper", "lower", "face") # Or ("face",) for the focused facial audit.
SPLITS = ("train", "val")        # Add "test" when ready for held-out evaluation.
MAX_SAMPLES_PER_SPLIT = None     # None = all; use 8 for a quick setup check.
SEED = 2342

MODE = "streaming"               # Frame-by-frame production codec path.
GEOMETRY = True                  # Requires the SMPL-X asset; False keeps parameter metrics.
WARMUP_FRAMES = 0                # Include startup errors; apply the same value to all splits.

# Paths resolve against REPO_ROOT. Empty means use the LM config's paths.
# Example: {"beatx_cache_path": "/data/beatx_gtdm3/database.hdf5"}
PATH_OVERRIDES = {}

# Results go in a new timestamped directory; checkpoints are never overwritten.
OUTPUT_PARENT = "reports/codec_reconstruction"


## Environment and checkpoint selection

The GPU selection must happen before CUDA is initialized. If this cell asks for a kernel restart, restart and run from the top. The current production codecs use their own model-factory precision; the notebook does not change them to match the gesture transformer's BF16 setting.

Only trusted local checkpoint files from the configured experiment are loaded. The data path should be the same `beatx_gtdm3` cache used by the teacher, rather than a differently cropped codec-training cache.


In [ ]:
import os
import sys
import json
from datetime import datetime, timezone

if REPO_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next((p for p in candidates
                      if (p / "miburi/models/gesture_codec.py").is_file()
                      and (p / "scripts/train.py").is_file()), None)
    if REPO_ROOT is None:
        raise RuntimeError("Set REPO_ROOT to the absolute miburi repository path.")
REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
if not (REPO_ROOT / "scripts/codec_reconstruction_audit.py").is_file():
    raise FileNotFoundError(f"Audit helper missing under {REPO_ROOT}; sync the notebook and helper files.")

if CUDA_VISIBLE_DEVICES is not None:
    loaded_torch = sys.modules.get("torch")
    if (loaded_torch is not None and loaded_torch.cuda.is_initialized()
            and os.environ.get("CUDA_VISIBLE_DEVICES") != str(CUDA_VISIBLE_DEVICES)):
        raise RuntimeError("CUDA is already initialized. Restart the kernel before changing GPU visibility.")
    os.environ["CUDA_VISIBLE_DEVICES"] = str(CUDA_VISIBLE_DEVICES)
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from scripts.codec_reconstruction_audit import AuditSettings, get_preflight, run_audit

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 15)
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select the training kernel/GPU, or set DEVICE='cpu'.")
print("Repository:", REPO_ROOT)
print("PyTorch:", torch.__version__, "| device:", DEVICE)
if DEVICE.startswith("cuda"):
    print("GPU:", torch.cuda.get_device_name(torch.device(DEVICE)))


In [ ]:
settings = AuditSettings(
    lm_config=LM_CONFIG,
    repository_root=str(REPO_ROOT),
    device=DEVICE,
    parts=PARTS,
    splits=SPLITS,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
    seed=SEED,
    mode=MODE,
    geometry=GEOMETRY,
    warmup_frames=WARMUP_FRAMES,
    path_overrides=PATH_OVERRIDES,
)
preflight = get_preflight(settings)
print(json.dumps(preflight, indent=2, default=str))


## Run the frozen reconstruction audit

The runner resets codec streaming state for every clip, uses canonical quantization in `eval()`, and verifies that model parameters and saved codebook buffers do not change. Finite clips with some `pose_valid=False` frames are evaluated on their complete original timeline. Required inputs containing NaN/Inf cause the whole clip to be skipped with its ID and reason recorded. Missing fields, incompatible shapes, read failures and model errors still stop the audit. No clip is replaced with another sample.

The default evaluates all selected clips at batch size one, without padding or random cropping. A sample cap limits selected clips; skipped clips are not replaced to fill the cap. Read the quality table before comparing results. Frame validity counts in its summary refer to evaluated clips; selected and skipped counts are also retained.

Three metric scopes are reported: `all_frames`, `valid_frames`, and `fully_valid_clips`. Valid-frame masking affects scoring only. Flagged inputs can still influence later codec states. Fully valid means every original frame was valid, before warmup exclusion; that subset may contain easier motions. Zero observations are reported as missing values, never perfect scores.

The manifest records selected IDs, quality findings, checkpoint hashes, settings, and overlapping source time intervals. An empty interval-overlap list does not prove that recordings are disjoint: non-overlapping chunks can still share a recording. Inspect source file IDs before interpreting a train/validation gap.


In [ ]:
# Display progress periodically, including skipped clips.
_progress_count = 0
def progress_update(*message):
    global _progress_count
    _progress_count += 1
    if _progress_count <= 3 or _progress_count % 25 == 0:
        print(*message)

result = run_audit(settings, progress=progress_update)
if result["metadata"].get("result_schema_version") != 2:
    raise RuntimeError("Sync both codec audit helper files, restart the kernel, and run from the top. The loaded helper returned an older result schema.")

def record_table(key, columns):
    frame = pd.DataFrame(result.get(key, []))
    # Keep a usable schema even when every selected clip is skipped.
    return frame.reindex(columns=list(dict.fromkeys([*columns, *frame.columns])))

summary = record_table("summary_records", ["split", "part", "metric_scope", "metric", "value", "count", "reduction", "n_samples", "mode"])
samples = record_table("sample_records", ["split", "part", "metric_scope", "metric", "value", "count", "reduction", "filechunk_id"])
codebooks = record_table("codebook_records", ["split", "part", "stage", "codebook_size", "token_count", "active_codes", "active_fraction", "entropy_nats", "perplexity", "unseen_train_mass", "usage_scope"])
code_counts = record_table("code_usage_records", ["split", "part", "stage", "code", "count", "usage_scope"])
quality_columns = ["split", "filechunk_id", "status", "reason", "frames", "valid_frames", "flagged_frames", "valid_fraction", "fully_valid", "nonfinite_fields"]
quality = record_table("quality_records", quality_columns)
skipped = record_table("skipped_records", quality_columns)
quality_summary = record_table("quality_summary_records", ["split", "available_clips", "selected_clips", "evaluated_clips", "skipped_clips", "fully_valid_clips", "flagged_clips", "selected_frames", "evaluated_frames", "valid_frames", "flagged_frames", "valid_fraction"])
summary["value"] = pd.to_numeric(summary["value"], errors="raise")
samples["value"] = pd.to_numeric(samples["value"], errors="raise")
print("Audit complete. Quality coverage by split:")
display(quality_summary)
if not skipped.empty:
    print("Skipped clips (full list is exported):")
    display(skipped.head(30))
if summary.empty:
    print("No reconstruction scores: inspect quality coverage and skipped reasons. Continue to export the report.")
print("Frozen state unchanged:", result["metadata"]["state_unchanged"])
print("The export cell saves the full metadata and per-clip quality records.")


## Reconstruction results

**Keep metric scopes separate:** `all_frames` scores every evaluated frame; `valid_frames` scores only valid frames; `fully_valid_clips` scores clips whose entire original validity mask is true. The same warmup exclusion applies to all scopes.

The summary pools metric numerators and counts before applying the final mean or square root. It does **not** average clip RMSEs, which would give short and long clips inappropriate relative weight.

- Rotations: mean geodesic error in **degrees**, evaluated on the actual reconstructed rotations.
- Expressions: coordinate RMSE in the dataset's coefficient units.
- Translation: errors in **metres**, plus decoded-velocity and reconstructed-trajectory velocity errors in **metres/second**. The trajectory follows the production codec's integration convention.
- Foot contacts: RMSE over the four contact channels in their native 0/1 target units, using the decoder's unrounded outputs.
- Optional SMPL-X geometry: joint errors in **millimetres**, and vertex displacement/velocity errors with explicit units. Face deformation is evaluated with the reconstructed jaw.
- Names containing `all_vertices` cover the complete SMPL-X vertex array under a face-only pose; they are **not** a lip-only or face-region metric. This averaging dilutes localized facial errors.

These metrics run at the native motion rate and use each sequence's own differences for velocity. They are intentionally not numerical replicas of the old `ReconMetrics` facial scores, which replace the reconstructed jaw and use a different velocity expression. FGD is not included: this audit measures direct reconstruction errors without loading a learned motion evaluator.

Trajectory and vertex velocities require both adjacent scored frames to be valid. The direct decoded-velocity target uses the production mixed finite differences; its valid-frame score also requires its source neighbors to be valid. No velocity pair bridges a masked gap.


In [ ]:
# Include metric_scope in the index so distinct evaluations cannot be averaged together.
if summary.empty:
    metric_table = pd.DataFrame(columns=list(SPLITS), index=pd.MultiIndex.from_arrays(
        [[], [], []], names=["metric_scope", "part", "metric"]))
else:
    metric_table = summary.set_index(["metric_scope", "part", "metric", "split"])["value"].unstack("split")
if "train" in metric_table and "val" in metric_table:
    metric_table["val_minus_train"] = metric_table["val"] - metric_table["train"]
    metric_table["val_over_train"] = metric_table["val"].div(
        metric_table["train"].where(metric_table["train"] > 1e-12))
display(metric_table.round(6))
display(summary[["split", "part", "metric_scope", "metric", "count", "reduction", "n_samples"]])


In [ ]:
# One facet per metric avoids mixing degrees, metres and coefficient units.
PLOT_SCOPE = "all_frames"  # Or "valid_frames" / "fully_valid_clips".
if PLOT_SCOPE not in ("all_frames", "valid_frames", "fully_valid_clips"):
    raise ValueError(f"Unknown metric scope: {PLOT_SCOPE}")
plot_rows = summary[summary["metric_scope"] == PLOT_SCOPE].dropna(subset=["value"]).copy()
available_pairs = list(plot_rows[["part", "metric"]].drop_duplicates().itertuples(index=False, name=None))
# Keep a readable first figure; the full table and CSV contain every metric.
preferred_words = ("rotation", "expression_rmse", "translation_rmse", "mpjpe",
                   "contact_rmse", "translation_velocity_direct", "coordinate_rmse_mm")
selected_pairs = [pair for pair in available_pairs if any(w in pair[1] for w in preferred_words)][:12]
if not selected_pairs:
    selected_pairs = available_pairs[:12]
ncols = min(3, max(1, len(selected_pairs)))
nrows = max(1, int(np.ceil(len(selected_pairs) / ncols)))
reconstruction_figure, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.5*nrows), squeeze=False)
palette = {"train": "#237b83", "val": "#c66b37", "test": "#776aab"}
from textwrap import fill
for ax, (part, metric) in zip(axes.flat, selected_pairs):
    rows = plot_rows[(plot_rows["part"] == part) & (plot_rows["metric"] == metric)]
    ax.bar(rows["split"], rows["value"], color=[palette.get(s, "#777777") for s in rows["split"]])
    ax.set_title(fill(f"{part}: {metric.replace('_', ' ')}", width=48), fontsize=9)
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", alpha=.2)
    ax.set_axisbelow(True)
if not selected_pairs:
    axes.flat[0].text(.5, .5, f"No scored observations for {PLOT_SCOPE}", ha="center", va="center")
    axes.flat[0].set_axis_off()
for ax in list(axes.flat)[max(1, len(selected_pairs)):]:
    ax.set_visible(False)
reconstruction_figure.suptitle(f"Frozen codec reconstruction: {PLOT_SCOPE}", fontsize=14)
reconstruction_figure.tight_layout(rect=(0, 0, 1, .97))
plt.show()


## RVQ code usage

`perplexity = exp(entropy)` describes the empirical distribution of **encoded ground-truth motion**, not transformer prediction perplexity. `active_fraction` is the fraction of codebook entries observed in this sample set. `unseen_train_mass` is the fraction of validation/test tokens assigned to codes absent from the audited training set.

A smaller validation set naturally visits fewer codes. Low usage on one speaker does not by itself establish codebook collapse, and maximizing usage is not a reconstruction objective. Compare distributions alongside reconstruction errors. When train is capped, “unseen” only means unseen in that capped subset.

Usage has scope `all_evaluated_frames`: every token from each complete evaluated clip is counted once, including tokens covering flagged frames and warmup. Skipped clips contribute no tokens. Code usage is not duplicated for the three metric scopes.


In [ ]:
usage_columns = ["split", "part", "stage", "codebook_size", "token_count",
                 "active_codes", "active_fraction", "entropy_nats", "perplexity", "unseen_train_mass", "usage_scope"]
display(codebooks.reindex(columns=usage_columns).sort_values(["part", "stage", "split"]).round(5))

usage_parts = list(codebooks["part"].drop_duplicates())
usage_figure, axes = plt.subplots(2, max(1, len(usage_parts)), figsize=(5*max(1, len(usage_parts)), 6.8), squeeze=False)
if not usage_parts:
    for ax in axes.flat:
        ax.text(.5, .5, "No evaluated tokens", ha="center", va="center")
        ax.set_axis_off()
for col, part in enumerate(usage_parts):
    for split in SPLITS:
        rows = codebooks[(codebooks["part"] == part) & (codebooks["split"] == split)].sort_values("stage")
        if rows.empty:
            continue
        for row, metric in enumerate(("perplexity", "active_fraction")):
            axes[row, col].plot(rows["stage"], rows[metric], marker="o", label=split,
                               color=palette.get(split))
            axes[row, col].set_title(f"{part}: {metric.replace('_', ' ')}")
            axes[row, col].set_xlabel("RVQ stage (1-based)")
            axes[row, col].set_xticks(sorted(codebooks.loc[codebooks["part"] == part, "stage"].unique()))
            axes[row, col].grid(alpha=.2)
            axes[row, col].legend(frameon=False)
    axes[1, col].set_ylim(0, 1.02)
usage_figure.suptitle("Code usage: all evaluated frames")
usage_figure.tight_layout(rect=(0, 0, 1, .96))
plt.show()


## Inspect difficult clips

Sort individual clips by a metric rather than relying on an aggregate alone. The table keeps the file/chunk IDs so that outliers can be traced back to their source recording. A clip with low reconstruction error can still have speech-ambiguous tokens; this audit does not measure speech-to-motion prediction.


In [ ]:
# Select one metric scope before ranking clips.
INSPECT_SCOPE = "all_frames"
if INSPECT_SCOPE not in ("all_frames", "valid_frames", "fully_valid_clips"):
    raise ValueError(f"Unknown metric scope: {INSPECT_SCOPE}")
scoped_samples = samples[samples["metric_scope"] == INSPECT_SCOPE]
if scoped_samples.empty:
    print(f"No clips with metrics for {INSPECT_SCOPE}; inspect the quality report.")
else:
    INSPECT_PART = "face" if "face" in set(scoped_samples["part"]) else scoped_samples["part"].iloc[0]
    INSPECT_SPLIT = "val" if "val" in set(scoped_samples["split"]) else scoped_samples["split"].iloc[0]
    options = scoped_samples[(scoped_samples["part"] == INSPECT_PART) & (scoped_samples["split"] == INSPECT_SPLIT)]
    metric_options = list(options["metric"].drop_duplicates())
    INSPECT_METRIC = next((m for m in metric_options if "expression_rmse" in m), metric_options[0])
    print("Available metrics:", metric_options)
    ranked = options[options["metric"] == INSPECT_METRIC].dropna(subset=["value"]).sort_values("value", ascending=False).head(20)
    display(ranked)
    display(quality[quality["filechunk_id"].isin(ranked["filechunk_id"]) & (quality["split"] == INSPECT_SPLIT)])


## Save a reproducible result

The export creates a new directory under `OUTPUT_PARENT` and writes the manifest, all result tables and figures. If you change settings or rerun an audit, rerun the result/plot cells before exporting. Keep a full-data baseline before adapting any codec.

The export also includes `quality_summary.csv`, `clip_quality.csv` and `skipped_clips.csv`. These are saved even if no reconstruction scores could be produced. Metric scopes remain separate in every reconstruction CSV; the manifest records which scope was plotted.


In [ ]:
export_stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
output_parent = Path(OUTPUT_PARENT).expanduser()
if not output_parent.is_absolute():
    output_parent = REPO_ROOT / output_parent
output_dir = output_parent / f"codec_audit_{export_stamp}"
output_dir.mkdir(parents=True, exist_ok=False)

summary.to_csv(output_dir / "reconstruction_summary.csv", index=False)
samples.to_csv(output_dir / "reconstruction_per_clip.csv", index=False)
codebooks.to_csv(output_dir / "codebook_summary.csv", index=False)
code_counts.to_csv(output_dir / "code_counts.csv", index=False)
metric_table.to_csv(output_dir / "train_validation_comparison.csv")
quality_summary.to_csv(output_dir / "quality_summary.csv", index=False)
quality.to_csv(output_dir / "clip_quality.csv", index=False)
skipped.to_csv(output_dir / "skipped_clips.csv", index=False)
manifest = {"exported_at_utc": datetime.now(timezone.utc).isoformat(),
            "preflight": preflight, "audit": result["metadata"], "plot_metric_scope": PLOT_SCOPE}
(output_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
for name, figure in (("reconstruction_metrics", reconstruction_figure), ("codebook_usage", usage_figure)):
    figure.savefig(output_dir / f"{name}.png", dpi=180, bbox_inches="tight")
    figure.savefig(output_dir / f"{name}.svg", bbox_inches="tight")
print("Saved audit:", output_dir.resolve())
display(pd.DataFrame({"file": [str(p.resolve()) for p in sorted(output_dir.iterdir())]}))


## How to use the evidence

| Observation | What it supports checking next |
| --- | --- |
| All-frame error is much higher than valid-frame error | Inspect flagged-frame prevalence and content; the flag is a whole-pose heuristic, not direct facial confidence. |
| Reconstruction is poor on both train and validation | Codec fit, capacity, preprocessing or domain mismatch; consider adaptation. |
| Train reconstruction is good but validation is much worse | Codec generalization and split composition; specialization can also worsen this gap. |
| Both reconstruct well, while transformer validation CE deteriorates | Prioritize the predictor and token predictability; good reconstruction alone does not prove an optimal tokenizer. |
| Validation frequently uses codes absent from a sufficiently large train audit | Inspect rare motion patterns and distribution shift before changing codebook size. |

A useful next experiment, if warranted, is to adapt **only the face codec** to Scott's training split while retaining four 2048-entry codebooks and the current time resolution. Re-audit its held-out reconstruction, then train a new gesture predictor using that tokenizer. Changing a tokenizer changes target distributions, so CE across different codecs is not a direct motion-quality comparison.

**Scope:** ground-truth reconstruction, not generated gestures, not teacher performance, and not evidence that the current codec has been trained exclusively on the audited split. The saved source-codec configs describe their historical setup; this notebook records the current evaluation split. No codec or teacher training is performed.

Compare like-for-like scopes and inspect coverage first. Fully valid subsets can differ in size and motion difficulty between splits. A low valid-frame error does not remove the effects of flagged inputs on subsequent causal state.
